# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR\^2 protein abundance dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the FAIR^2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and their fields by @id
record_sets = list(metadata.record_sets)  # List of RecordSet objects
print('Available RecordSets:')
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}, label: {rs.label if hasattr(rs, 'label') else ''}")
    if hasattr(rs, 'fields') and rs.fields:
        for fld in rs.fields:
            print(f"    Field @id: {fld.id}, label: {fld.label if hasattr(fld, 'label') else ''}, dataType: {fld.data_type if hasattr(fld, 'data_type') else ''}")

## 3. Data Extraction
Load data from specific record set(s) into Pandas DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this FAIR^2 dataset, the main record set (table of protein measurements) is usually called 'ProteinTable' or similar.
# Let's select the first record set for demonstration. You can adjust this after reviewing the above overview.

# WARNING: Update record_set_ids after inspecting printed RecordSet @ids above if desired.
record_set_ids = [rs.id for rs in record_sets]
print('RecordSet @ids for extraction:', record_set_ids)

dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
    except Exception as e:
        print(f"Could not load records for RecordSet {record_set_id}: {e}")

# Choose the primary RecordSet for demonstration (likely the main data table)
main_record_set_id = record_set_ids[0] if record_set_ids else None

if main_record_set_id and main_record_set_id in dataframes:
    print(f"Columns for RecordSet {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No data loaded! Check record set IDs and data availability.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
import numpy as np

# Identify a numeric field for demonstration (e.g., 'MolecularWeight' or 'Abundance')
df = dataframes[main_record_set_id] if main_record_set_id and main_record_set_id in dataframes else None

# Update these variable names as appropriate after viewing actual dataframe columns above
numeric_field = None
for col in (df.columns if df is not None else []):
    # Look for likely numeric columns
    if 'abundance' in col.lower() or 'molecularweight' in col.lower() or 'mw' in col.lower() or 'count' in col.lower() or 'peptide' in col.lower():
        numeric_field = col
        break
if not numeric_field and df is not None:
    # Fallback to first numeric dtype
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    if num_cols:
        numeric_field = num_cols[0]

print(f"Using numeric field for EDA: {numeric_field}")

# Example threshold
threshold = None
if numeric_field and df is not None and numeric_field in df.columns:
    # Compute a simple threshold as e.g. 10th percentile
    threshold = df[numeric_field].quantile(0.25) if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (showing 5):")
    display(filtered_df.head())

    # Normalize the numeric field
    mean = filtered_df[numeric_field].mean()
    std = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field (e.g., 'description' or 'accession')
    group_field = None
    for col in df.columns:
        if col != numeric_field and (df[col].dtype == object) and df[col].nunique() < 20:
            group_field = col
            break
    print(f"Grouping field: {group_field}")

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped means of {numeric_field} by {group_field} (showing up to 5):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        order = df[group_field].value_counts().index
        sns.boxplot(x=group_field, y=numeric_field, data=df, order=order)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Explored dataset metadata and structure using Croissant schema and `mlcroissant`.
- Loaded main protein record set and demonstrated selection, normalization, and simple aggregation/grouping of numeric fields.
- Visualized field distributions and group-wise statistics.
- For further biological, clinical, or statistical analysis, refine field selections and consider domain-specific questions using the provided protein measurements.